# Evals en el SDLC con IA

## Cómo dejar de adivinar y empezar a medir

---

**Workshop NTT Data · Anthropic Partner Basecamp**

*Claude Evals Workshop — Lab 00: Introducción*

# El problema: "vibe coding"

## ¿Qué es el drift de intención?

- Pides a Claude que escriba una función → parece funcionar ✓
- Cambias el prompt ligeramente → la salida cambia sin avisar
- En producción, nadie sabe si el comportamiento es correcto o simplemente *plausible*

---

## El resultado

> "Funciona en mis pruebas manuales"
> — Todo el mundo, antes del incidente en producción

**Sin medición sistemática, el feedback loop es: ojo humano → intuición → deploy**

# Qué es un evaluador (eval)

## Definición

Un **evaluador** es una función que toma la salida de un LLM y devuelve una señal de calidad medible.

```
evaluate(output, expected) → score | pass/fail
```

---

## Analogía con testing

| Testing tradicional | Evaluador de LLM |
|---------------------|-----------------|
| `assert resultado == esperado` | `score = evaluar(respuesta_llm)` |
| Determinista | Puede ser probabilístico |
| Falla o pasa | Puede tener gradiente (0.0 – 1.0) |
| CI lo ejecuta automáticamente | CI lo ejecuta automáticamente |

**Los evaluadores son tests para comportamiento no determinista.**

# Tipo 1: Exact Match

## Comparación literal, 100% determinista

```python
def exact_match(output: str, expected: str) -> bool:
    return output.strip() == expected.strip()
```

---

## Cuándo usarlo

- Extracción de campos estructurados (JSON, fechas, IDs)
- Clasificación con etiquetas fijas ("positivo" / "negativo")
- Respuestas de una sola palabra o número

## Limitaciones

- No tolera variación legítima de lenguaje natural
- Frágil ante cambios de formato triviales (mayúsculas, espacios)

> **Regla de oro**: si la respuesta correcta tiene exactamente una forma posible, usa Exact Match.

# Tipo 2: Rule-Based

## Lógica Python, sin LLM

```python
def rule_based_check(output: str) -> float:
    score = 0.0
    if len(output) > 50:                    score += 0.25
    if "```" in output:                     score += 0.25  # tiene código
    if output.count("\n") >= 3:            score += 0.25  # tiene estructura
    if not output.startswith("Lo siento"):  score += 0.25
    return score
```

---

## Cuándo usarlo

- Verificar presencia de elementos obligatorios (citas, formato, secciones)
- Validar restricciones de longitud o estructura
- Cualquier regla que se pueda expresar con lógica Python

## Ventajas

- Rápido, barato, reproducible
- Sin dependencia de red ni coste de tokens

# Tipo 3: LLM-as-Judge

## Claude evalúa la calidad

```python
judge_prompt = f"""
Evalúa la siguiente respuesta de un asistente de código.
Puntúa de 0 a 10 en: corrección, claridad y seguridad.
Responde solo con JSON: {{"score": X, "razon": "..."}}

Respuesta a evaluar:
{output}
"""
result = claude(judge_prompt)
```

---

## Cuándo usarlo

- Calidad subjetiva: tono, claridad, adecuación al contexto
- Cuando "correcto" tiene múltiples formas válidas
- Evaluación de razonamiento o explicaciones

## Consideraciones

- Más lento y costoso que los otros tipos
- Requiere meta-evaluación: ¿el juez es fiable?
- Usar con prompt de juez cuidadosamente diseñado y testeado

# El ciclo evaluación-driven

```
┌─────────────────────────────────────────────────┐
│                                                 │
│   BASELINE          CAMBIO          MEDIR       │
│                                                 │
│  Mide calidad   →  Modifica  →  Mide calidad   │
│  en v_actual       prompt/       en v_nueva     │
│                    modelo/                      │
│                    código                       │
│                       ↑                         │
│                    ITERAR                       │
│               (si score no mejora)              │
│                                                 │
└─────────────────────────────────────────────────┘
```

---

## El principio fundamental

> No merges sin métricas.
> Si no tienes un criterio medible de "listo", no tienes un criterio de "listo".

**Esto es lo que los 4 labs del workshop ponen en práctica.**

# Los 4 labs del workshop

| Lab | Título | Qué aprendes |
|-----|--------|--------------|
| **01** | SDLC Gatekeeper | Evaluadores como gate en CI/CD con GitHub Actions |
| **02** | Architect Agent | Medir razonamiento de un agente de arquitectura |
| **03** | Performance | Medir TTFT, TTC, tokens y coste por modelo |
| **04** | Pipeline E2E | Combinar los 3 labs en un pipeline CI/CD automatizado |

---

## Flujo del workshop

```
Lab 00 (conceptos)
    → Lab 01 (primer evaluador real en CI)
        → Lab 02 (agente + evaluación de razonamiento)
            → Lab 03 (métricas de performance)
                → Lab 04 (pipeline end-to-end)
```

Cada lab es autocontenido — puedes ejecutarlo de forma independiente.

# Preguntas

---

## Recursos

- **Documentación Anthropic**: [docs.anthropic.com/en/docs/test-evaluate-prompts](https://docs.anthropic.com/en/docs/test-evaluate-prompts)
- **Este workshop**: carpeta `labs/` en el repositorio
- **Spec de diseño**: `docs/superpowers/specs/2026-05-20-evals-workshop-design.md`

---

*Claude Evals Workshop — NTT Data · Anthropic Partner Basecamp*